In [47]:
import sys
print(sys.executable)

c:\Users\sebdy\AI Analytics Copilot API\.venv\Scripts\python.exe


In [48]:
import pandas as pd
import numpy as np
from google.cloud import bigquery

In [49]:
from pathlib import Path
csv_path = Path('..\data\conjura_mmm_data.csv')
df = pd.read_csv(csv_path)
df.head()

,MMM_TIMESERIES_ID,ORGANISATION_ID,ORGANISATION_VERTICAL,ORGANISATION_SUBVERTICAL,ORGANISATION_MARKETING_SOURCES,ORGANISATION_PRIMARY_TERRITORY_NAME,TERRITORY_NAME,DATE_DAY,CURRENCY_CODE,FIRST_PURCHASES,...,META_FACEBOOK_IMPRESSIONS,META_INSTAGRAM_IMPRESSIONS,META_OTHER_IMPRESSIONS,TIKTOK_IMPRESSIONS,DIRECT_CLICKS,BRANDED_SEARCH_CLICKS,ORGANIC_SEARCH_CLICKS,EMAIL_CLICKS,REFERRAL_CLICKS,ALL_OTHER_CLICKS
0,596eef7c71f933d820d0e485935d0e8f,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,"Google, Meta",US,All Territories,2022-07-29,USD,22,...,18997.0,NaN,NaN,NaN,139.0,NaN,300.0,1.0,61.0,40.0
1,596eef7c71f933d820d0e485935d0e8f,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,"Google, Meta",US,All Territories,2022-07-30,USD,14,...,20188.0,NaN,NaN,NaN,209.0,NaN,442.0,8.0,110.0,62.0
2,596eef7c71f933d820d0e485935d0e8f,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,"Google, Meta",US,All Territories,2022-07-31,USD,31,...,24718.0,NaN,NaN,NaN,262.0,NaN,427.0,631.0,108.0,65.0
3,596eef7c71f933d820d0e485935d0e8f,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,"Google, Meta",US,All Territories,2022-08-01,USD,18,...,25076.0,NaN,NaN,NaN,247.0,NaN,400.0,117.0,125.0,68.0
4,596eef7c71f933d820d0e485935d0e8f,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,"Google, Meta",US,All Territories,2022-08-02,USD,23,...,22688.0,NaN,NaN,NaN,255.0,NaN,425.0,37.0,146.0,65.0


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132759 entries, 0 to 132758
Data columns (total 50 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   MMM_TIMESERIES_ID                    132759 non-null  object 
 1   ORGANISATION_ID                      132759 non-null  object 
 2   ORGANISATION_VERTICAL                124649 non-null  object 
 3   ORGANISATION_SUBVERTICAL             124649 non-null  object 
 4   ORGANISATION_MARKETING_SOURCES       132759 non-null  object 
 5   ORGANISATION_PRIMARY_TERRITORY_NAME  132759 non-null  object 
 6   TERRITORY_NAME                       132759 non-null  object 
 7   DATE_DAY                             132759 non-null  object 
 8   CURRENCY_CODE                        132759 non-null  object 
 9   FIRST_PURCHASES                      132759 non-null  int64  
 10  FIRST_PURCHASES_UNITS                132759 non-null  int64  
 11  FIRST_PURCHAS

In [51]:
df['DATE_DAY'] = pd.to_datetime(df['DATE_DAY'])

In [52]:
null_df = pd.DataFrame(df.isna().sum(axis=0).sort_values(ascending=False))
null_df.rename(columns={0: 'nulls'}, inplace=True)
null_df['%null'] = 100*null_df['nulls']/len(df)
null_df

,nulls,%null
TIKTOK_CLICKS,129297,97.392267
TIKTOK_SPEND,129297,97.392267
TIKTOK_IMPRESSIONS,129297,97.392267
GOOGLE_VIDEO_SPEND,125131,94.254250
GOOGLE_VIDEO_IMPRESSIONS,125131,94.254250
GOOGLE_VIDEO_CLICKS,125131,94.254250
GOOGLE_DISPLAY_SPEND,114013,85.879677
GOOGLE_DISPLAY_CLICKS,114013,85.879677
GOOGLE_DISPLAY_IMPRESSIONS,114013,85.879677
META_OTHER_CLICKS,109214,82.264856


In [53]:
discard_cols = list(null_df[null_df['%null']>80].index)
df.drop(discard_cols, axis=1, inplace=True)
len(df.columns)

38

In [54]:
spend_cols = [
    'GOOGLE_PAID_SEARCH_SPEND',
    'GOOGLE_SHOPPING_SPEND',
    'GOOGLE_PMAX_SPEND',
    'META_FACEBOOK_SPEND',
    'META_INSTAGRAM_SPEND',
    ]

clicks_cols = [
    'GOOGLE_PAID_SEARCH_CLICKS',
    'GOOGLE_SHOPPING_CLICKS',
    'GOOGLE_PMAX_CLICKS',
    'META_FACEBOOK_CLICKS',
    'META_INSTAGRAM_CLICKS'
]

impression_cols = [
    'GOOGLE_PAID_SEARCH_IMPRESSIONS',
    'GOOGLE_SHOPPING_IMPRESSIONS',
    'GOOGLE_PMAX_IMPRESSIONS',
    'META_FACEBOOK_IMPRESSIONS',
    'META_INSTAGRAM_IMPRESSIONS'
]

In [55]:
channel_map = {
    'google_paid_search': {
        'spend': 'GOOGLE_PAID_SEARCH_SPEND',
        'clicks': 'GOOGLE_PAID_SEARCH_CLICKS',
        'impressions': 'GOOGLE_PAID_SEARCH_IMPRESSIONS'
    },
    'google_shopping': {
        'spend': 'GOOGLE_SHOPPING_SPEND',
        'clicks': 'GOOGLE_SHOPPING_CLICKS',
        'impressions': 'GOOGLE_SHOPPING_IMPRESSIONS'
    },
    'google_pmax': {
        'spend': 'GOOGLE_PMAX_SPEND',
        'clicks': 'GOOGLE_PMAX_CLICKS',
        'impressions': 'GOOGLE_PMAX_IMPRESSIONS'
    },
    'meta_facebook': {
        'spend': 'META_FACEBOOK_SPEND',
        'clicks': 'META_FACEBOOK_CLICKS',
        'impressions': 'META_FACEBOOK_IMPRESSIONS'
    },
    'meta_instagram': {
        'spend': 'META_INSTAGRAM_SPEND',
        'clicks': 'META_INSTAGRAM_CLICKS',
        'impressions': 'META_INSTAGRAM_IMPRESSIONS'
    }
}

In [56]:
base_cols = [
    'DATE_DAY',
    'ORGANISATION_ID',
    'ORGANISATION_VERTICAL',
    'ORGANISATION_SUBVERTICAL',
    'ORGANISATION_PRIMARY_TERRITORY_NAME',
    'TERRITORY_NAME',
    'CURRENCY_CODE',
    'ALL_PURCHASES',
    'ALL_PURCHASES_ORIGINAL_PRICE'
]

channel_frames = []

for channel, cols in channel_map.items():
    temp = df[base_cols].copy()
    temp['channel'] = channel
    temp['spend'] = df[cols['spend']].fillna(0)
    temp['clicks'] = df[cols['clicks']].fillna(0)
    temp['impressions'] = df[cols['impressions']].fillna(0)
    channel_frames.append(temp)

channel_df = pd.concat(channel_frames, ignore_index=True)

In [57]:
channel_df['ctr'] = np.where(
    channel_df['impressions'] > 0,
    channel_df['clicks'] / channel_df['impressions'],
    np.nan
)

channel_df['cpc'] = np.where(
    channel_df['clicks'] > 0,
    channel_df['spend'] / channel_df['clicks'],
    np.nan
)

channel_df['cpm'] = np.where(
    channel_df['impressions'] > 0,
    channel_df['spend'] / channel_df['impressions'] * 1000,
    np.nan
)

In [58]:
channel_df = channel_df[
    (channel_df['spend'] > 0) |
    (channel_df['clicks'] > 0) |
    (channel_df['impressions'] > 0)
].copy()

In [59]:
channel_df.head()

,DATE_DAY,ORGANISATION_ID,ORGANISATION_VERTICAL,ORGANISATION_SUBVERTICAL,ORGANISATION_PRIMARY_TERRITORY_NAME,TERRITORY_NAME,CURRENCY_CODE,ALL_PURCHASES,ALL_PURCHASES_ORIGINAL_PRICE,channel,spend,clicks,impressions,ctr,cpc,cpm
3,2022-08-01,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,22,3335.970202,google_paid_search,35.24,48.0,890.0,0.053933,0.734167,39.595506
4,2022-08-02,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,28,3991.987441,google_paid_search,38.40,46.0,702.0,0.065527,0.834783,54.700855
5,2022-08-03,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,25,4513.998777,google_paid_search,25.13,31.0,356.0,0.087079,0.810645,70.589888
6,2022-08-04,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,24,3756.990457,google_paid_search,20.57,23.0,373.0,0.061662,0.894348,55.147453
7,2022-08-05,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,38,7620.990937,google_paid_search,17.34,22.0,436.0,0.050459,0.788182,39.770642


In [60]:
channel_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 343339 entries, 3 to 663147
Data columns (total 16 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   DATE_DAY                             343339 non-null  datetime64[ns]
 1   ORGANISATION_ID                      343339 non-null  object        
 2   ORGANISATION_VERTICAL                318094 non-null  object        
 3   ORGANISATION_SUBVERTICAL             318094 non-null  object        
 4   ORGANISATION_PRIMARY_TERRITORY_NAME  343339 non-null  object        
 5   TERRITORY_NAME                       343339 non-null  object        
 6   CURRENCY_CODE                        343339 non-null  object        
 7   ALL_PURCHASES                        343339 non-null  int64         
 8   ALL_PURCHASES_ORIGINAL_PRICE         343339 non-null  float64       
 9   channel                              343339 non-null  object        
 10  s

In [61]:
summary_df = df.copy()

summary_df['total_paid_spend'] = summary_df[spend_cols].fillna(0).sum(axis=1)
summary_df['total_paid_clicks'] = summary_df[clicks_cols].fillna(0).sum(axis=1)
summary_df['total_paid_impressions'] = summary_df[impression_cols].fillna(0).sum(axis=1)

organic_click_cols = [
    'DIRECT_CLICKS',
    'BRANDED_SEARCH_CLICKS',
    'ORGANIC_SEARCH_CLICKS',
    'EMAIL_CLICKS',
    'REFERRAL_CLICKS',
    'ALL_OTHER_CLICKS'
]

summary_df[organic_click_cols] = summary_df[organic_click_cols].fillna(0)

summary_df['ORGANISATION_VERTICAL'] = summary_df['ORGANISATION_VERTICAL'].fillna('Unknown')
summary_df['ORGANISATION_SUBVERTICAL'] = summary_df['ORGANISATION_SUBVERTICAL'].fillna('Unknown')

summary_df['total_paid_ctr'] = np.where(
    summary_df['total_paid_impressions'] > 0,
    summary_df['total_paid_clicks'] / summary_df['total_paid_impressions'],
    np.nan
)

summary_df['total_paid_cpc'] = np.where(
    summary_df['total_paid_clicks'] > 0,
    summary_df['total_paid_spend'] / summary_df['total_paid_clicks'],
    np.nan
)

summary_df['avg_order_value'] = np.where(
    summary_df['ALL_PURCHASES'] > 0,
    summary_df['ALL_PURCHASES_ORIGINAL_PRICE'] / summary_df['ALL_PURCHASES'],
    np.nan
)

summary_df['discount_rate'] = np.where(
    summary_df['ALL_PURCHASES_ORIGINAL_PRICE'] > 0,
    summary_df['ALL_PURCHASES_GROSS_DISCOUNT'] / summary_df['ALL_PURCHASES_ORIGINAL_PRICE'],
    np.nan
)

summary_df['roas_like'] = np.where(
    summary_df['total_paid_spend'] > 0,
    summary_df['ALL_PURCHASES_ORIGINAL_PRICE'] / summary_df['total_paid_spend'],
    np.nan
)

summary_df = summary_df[
    [
        'DATE_DAY',
        'ORGANISATION_ID',
        'ORGANISATION_VERTICAL',
        'ORGANISATION_SUBVERTICAL',
        'ORGANISATION_PRIMARY_TERRITORY_NAME',
        'TERRITORY_NAME',
        'CURRENCY_CODE',
        'FIRST_PURCHASES',
        'ALL_PURCHASES',
        'ALL_PURCHASES_UNITS',
        'ALL_PURCHASES_ORIGINAL_PRICE',
        'ALL_PURCHASES_GROSS_DISCOUNT',
        'DIRECT_CLICKS',
        'BRANDED_SEARCH_CLICKS',
        'ORGANIC_SEARCH_CLICKS',
        'EMAIL_CLICKS',
        'REFERRAL_CLICKS',
        'ALL_OTHER_CLICKS',
        'total_paid_spend',
        'total_paid_clicks',
        'total_paid_impressions',
        'total_paid_ctr',
        'total_paid_cpc',
        'avg_order_value',
        'discount_rate',
        'roas_like'
    ]
].copy()


In [62]:
summary_df.head()

,DATE_DAY,ORGANISATION_ID,ORGANISATION_VERTICAL,ORGANISATION_SUBVERTICAL,ORGANISATION_PRIMARY_TERRITORY_NAME,TERRITORY_NAME,CURRENCY_CODE,FIRST_PURCHASES,ALL_PURCHASES,ALL_PURCHASES_UNITS,...,REFERRAL_CLICKS,ALL_OTHER_CLICKS,total_paid_spend,total_paid_clicks,total_paid_impressions,total_paid_ctr,total_paid_cpc,avg_order_value,discount_rate,roas_like
0,2022-07-29,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,22,27,32,...,61.0,40.0,439.278905,418.0,50904.0,0.008212,1.050906,168.629046,0.185942,10.364678
1,2022-07-30,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,14,17,26,...,110.0,62.0,525.922025,476.0,64671.0,0.007360,1.104878,186.941061,0.193143,6.042717
2,2022-07-31,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,31,39,48,...,108.0,65.0,701.946429,553.0,82891.0,0.006671,1.269343,140.230739,0.275187,7.791191
3,2022-08-01,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,18,22,34,...,125.0,68.0,652.532798,531.0,81525.0,0.006513,1.228875,151.635009,0.226144,5.112341
4,2022-08-02,04769dac8b828ec7a85676d9e2bffe6f,Beauty & Fitness,Hair Care,US,All Territories,USD,23,28,33,...,146.0,65.0,610.972971,503.0,71244.0,0.007060,1.214658,142.570980,0.233971,6.533820


In [63]:
summary_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132759 entries, 0 to 132758
Data columns (total 26 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   DATE_DAY                             132759 non-null  datetime64[ns]
 1   ORGANISATION_ID                      132759 non-null  object        
 2   ORGANISATION_VERTICAL                132759 non-null  object        
 3   ORGANISATION_SUBVERTICAL             132759 non-null  object        
 4   ORGANISATION_PRIMARY_TERRITORY_NAME  132759 non-null  object        
 5   TERRITORY_NAME                       132759 non-null  object        
 6   CURRENCY_CODE                        132759 non-null  object        
 7   FIRST_PURCHASES                      132759 non-null  int64         
 8   ALL_PURCHASES                        132759 non-null  int64         
 9   ALL_PURCHASES_UNITS                  132759 non-null  int64         
 

In [64]:
from pathlib import Path

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(processed_dir / 'business_daily_summary.csv', index=False)
channel_df.to_csv(processed_dir / 'channel_daily_performance.csv', index=False)